# Linear Regression: Simple and Multiple Regression
**Datasets:** Wheat Yield (time series) · Medical Insurance Charges  
**Methods:** Simple Linear Regression · Multiple Linear Regression · Feature Engineering · Interaction Terms

---

## Overview

Linear regression is one of the most foundational tools in data science and statistics. This notebook covers two variants:

- **Simple linear regression** — modeling a single input feature against a continuous output (wheat yield over time)
- **Multiple linear regression (MLR)** — modeling multiple features simultaneously to predict a target (predicting medical insurance charges from age, BMI, smoking status, and more)

We also explore **feature engineering** — creating new input variables (like interaction terms) to improve model performance — and evaluate models using R², MAE, and MSE.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Simple Linear Regression — Wheat Yield Over Time

We start with a classic simple linear regression problem: predicting annual wheat yield from the year number. This shows the core sklearn workflow — fit a model, inspect coefficients, and generate predictions.

**Dataset:** 23 years of U.S. wheat yield data (metric tons).

In [ ]:
# Annual wheat yield data — one value per year, starting at year 1
yield_wheat = [
    4149989, 4231117, 4429500, 4531384, 4500052, 4554976,
    4755886, 4740432, 4823722, 4835339, 5055926, 5069224,
    5109104, 5340733, 5224175, 5250805, 5287310, 5670746,
    5493230, 5506444, 5794320, 5784075, 6019956
]
years = list(range(1, 24))  # Year numbers 1–23

print(f"Number of observations: {len(years)}")
print(f"Year range: {years[0]}–{years[-1]}")
print(f"Yield range: {min(yield_wheat):,} – {max(yield_wheat):,} metric tons")

In [ ]:
# Visualize the raw data before modeling
plt.figure(figsize=(10, 5))
plt.plot(years, yield_wheat, 'o', color='steelblue', markersize=8, label='Observed yield')
plt.xlabel('Year')
plt.ylabel('Wheat Yield (metric tons)')
plt.title('Annual Wheat Yield Over Time')
plt.legend()
plt.tight_layout()
plt.show()

### 2.1 Fit the model

sklearn's `LinearRegression` expects features as a 2D array (matrix) even for a single input.
We reshape `years` from shape `(23,)` to `(23, 1)` using `.reshape(-1, 1)`.

The fitted model gives us the **slope** (`coef_`) and **intercept** (`intercept_`), defining the line:

> **yield = intercept + slope × year**

In [ ]:
# Reshape X to a column vector — sklearn requires 2D input
X = np.array(years).reshape(-1, 1)
y = np.array(yield_wheat)

# Instantiate and fit the model
model_slr = LinearRegression()
model_slr.fit(X, y)

print(f"Slope (coefficient): {model_slr.coef_[0]:,.2f}")
print(f"Intercept:           {model_slr.intercept_:,.2f}")
print()
print(f"Interpretation: each additional year, wheat yield increases by ~{model_slr.coef_[0]:,.0f} metric tons")

In [ ]:
# Generate predicted values and plot the regression line
y_pred_slr = model_slr.predict(X)

plt.figure(figsize=(10, 5))
plt.plot(years, y, 'o', color='steelblue', markersize=8, label='Observed yield')
plt.plot(years, y_pred_slr, color='crimson', linewidth=2, label='Linear regression fit')
plt.xlabel('Year')
plt.ylabel('Wheat Yield (metric tons)')
plt.title('Wheat Yield — Linear Regression Fit')
plt.legend()
plt.tight_layout()
plt.show()

r2 = model_slr.score(X, y)
print(f"R² (coefficient of determination): {r2:.4f}")
print(f"The model explains {r2*100:.1f}% of the variance in wheat yield")

In [ ]:
# Predict year 24 (the next year, not in the training data)
year_24 = np.array([[24]])
predicted_year_24 = model_slr.predict(year_24)
print(f"Predicted wheat yield for year 24: {predicted_year_24[0]:,.0f} metric tons")

## 3. Multiple Linear Regression — Insurance Charges

Real-world predictions rarely depend on just one variable. Multiple linear regression extends the same framework to handle many features at once:

> **charges = β₀ + β₁·age + β₂·bmi + β₃·smoker + β₄·children + ...**

We'll use a medical insurance dataset to predict annual charges based on demographic and lifestyle features.

In [ ]:
# Download the insurance training and test datasets
!gdown 1I6pl-QXxUoHE04fJbUI8_8_ZDF6H4LLx   # insurance_training.csv
!gdown 1ahaVYrLemIFtAb9sKe4xisIC_aqA6Ecx   # insurance_test.csv

In [ ]:
# Load and inspect the training data
df_train = pd.read_csv('insurance_training.csv')

print(f"Shape: {df_train.shape}")
print(f"Columns: {list(df_train.columns)}")
df_train.head()

In [ ]:
# Summary statistics — understand the distributions
df_train.describe()

### 3.1 Exploratory Data Analysis

Before building a model, we explore the data to understand which variables are likely to be strong predictors of insurance charges.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age vs charges
axes[0, 0].scatter(df_train['age'], df_train['charges'], alpha=0.4, color='steelblue')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Charges ($)')
axes[0, 0].set_title('Age vs. Insurance Charges')

# BMI vs charges
axes[0, 1].scatter(df_train['bmi'], df_train['charges'], alpha=0.4, color='darkorange')
axes[0, 1].set_xlabel('BMI')
axes[0, 1].set_ylabel('Charges ($)')
axes[0, 1].set_title('BMI vs. Insurance Charges')

# Charges by sex
df_train.groupby('sex')['charges'].mean().plot(kind='bar', ax=axes[1, 0], color=['#4C72B0', '#DD8452'])
axes[1, 0].set_title('Mean Charges by Sex')
axes[1, 0].set_ylabel('Mean Charges ($)')
axes[1, 0].tick_params(axis='x', rotation=0)

# Charges by smoker status
df_train.groupby('smoker')['charges'].mean().plot(kind='bar', ax=axes[1, 1], color=['#55A868', '#C44E52'])
axes[1, 1].set_title('Mean Charges by Smoker Status')
axes[1, 1].set_ylabel('Mean Charges ($)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle('Insurance Charges — Exploratory Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Who pays the most? Who pays the least?
highest_payer = df_train.loc[df_train['charges'].idxmax()]
lowest_payer  = df_train.loc[df_train['charges'].idxmin()]

print("Highest charges:")
print(highest_payer)
print()
print("Lowest charges:")
print(lowest_payer)

In [ ]:
# Correlation heatmap — requires encoding categorical variables first
df_encoded = df_train.copy()
df_encoded['sex_enc']    = (df_encoded['sex'] == 'male').astype(int)
df_encoded['smoker_enc'] = (df_encoded['smoker'] == 'yes').astype(int)

corr_cols = ['age', 'bmi', 'children', 'sex_enc', 'smoker_enc', 'charges']
corr_matrix = df_encoded[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("Key finding: smoker status has by far the strongest correlation with charges")

### 3.2 Baseline Multiple Linear Regression

We start with four features: `age`, `bmi`, `children`, and `smoker` (encoded as 0/1).

Note: `smoker` is a categorical variable ('yes'/'no') and must be converted to a number before sklearn can use it.

In [ ]:
# Encode smoker as binary (1 = smoker, 0 = non-smoker)
df_train['smoker_enc'] = (df_train['smoker'] == 'yes').astype(int)

# Define features and target
feature_cols = ['age', 'bmi', 'children', 'smoker_enc']
X_train = df_train[feature_cols]
y_train = df_train['charges']

# Fit the baseline model
model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)

# Evaluate on training data
r2_baseline = model_baseline.score(X_train, y_train)
y_pred_baseline = model_baseline.predict(X_train)
mae_baseline = mean_absolute_error(y_train, y_pred_baseline)
mse_baseline = mean_squared_error(y_train, y_pred_baseline)

print("Baseline MLR — Training Performance")
print(f"  R²:  {r2_baseline:.4f}")
print(f"  MAE: ${mae_baseline:,.2f}  (average error in dollars)")
print(f"  MSE: {mse_baseline:,.0f}")
print()
print("Coefficients:")
for feat, coef in zip(feature_cols, model_baseline.coef_):
    print(f"  {feat:<15}: {coef:>10,.2f}")
print(f"  {'intercept':<15}: {model_baseline.intercept_:>10,.2f}")

In [ ]:
# Visualize predicted vs actual charges
plt.figure(figsize=(8, 6))
plt.scatter(y_train, y_pred_baseline, alpha=0.4, color='steelblue')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
         'r--', linewidth=2, label='Perfect prediction line')
plt.xlabel('Actual Charges ($)')
plt.ylabel('Predicted Charges ($)')
plt.title(f'Baseline MLR — Predicted vs Actual\n(R² = {r2_baseline:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

### 3.3 Feature Engineering — Interaction Terms

The baseline model assumes each feature affects charges independently. But in practice, features often interact:

- **Smoking + high BMI** might be more expensive than the sum of each separately
- **Being older and a smoker** might dramatically increase risk

Interaction terms capture these combined effects by multiplying pairs of features together. We create all pairwise combinations of our four base features.

In [ ]:
# Build feature matrix with interaction terms
X_interact = df_train[feature_cols].copy()

# All pairwise interactions
X_interact['age_x_bmi']          = df_train['age'] * df_train['bmi']
X_interact['age_x_smoker']       = df_train['age'] * df_train['smoker_enc']
X_interact['age_x_children']     = df_train['age'] * df_train['children']
X_interact['bmi_x_smoker']       = df_train['bmi'] * df_train['smoker_enc']
X_interact['bmi_x_children']     = df_train['bmi'] * df_train['children']
X_interact['smoker_x_children']  = df_train['smoker_enc'] * df_train['children']

print(f"Feature matrix shape: {X_interact.shape}")
print(f"Features: {list(X_interact.columns)}")

In [ ]:
# Fit the extended model with interactions
model_interact = LinearRegression()
model_interact.fit(X_interact, y_train)

r2_interact = model_interact.score(X_interact, y_train)
y_pred_interact = model_interact.predict(X_interact)
mae_interact = mean_absolute_error(y_train, y_pred_interact)
mse_interact = mean_squared_error(y_train, y_pred_interact)

print("MLR with Interaction Terms — Training Performance")
print(f"  R²:  {r2_interact:.4f}  (vs {r2_baseline:.4f} baseline)")
print(f"  MAE: ${mae_interact:,.2f}  (vs ${mae_baseline:,.2f} baseline)")
print(f"  MSE: {mse_interact:,.0f}")

In [ ]:
# Compare the two models visually
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline
axes[0].scatter(y_train, y_pred_baseline, alpha=0.4, color='steelblue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Charges ($)')
axes[0].set_ylabel('Predicted Charges ($)')
axes[0].set_title(f'Baseline MLR\nR² = {r2_baseline:.3f}')

# Interaction model
axes[1].scatter(y_train, y_pred_interact, alpha=0.4, color='darkorange')
axes[1].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Charges ($)')
axes[1].set_ylabel('Predicted Charges ($)')
axes[1].set_title(f'MLR with Interactions\nR² = {r2_interact:.3f}')

plt.suptitle('Predicted vs Actual — Baseline vs Interaction Model', fontsize=13)
plt.tight_layout()
plt.show()

### 3.4 Evaluation on the Test Set

Training performance can be misleading — a complex model can fit the training data well but fail on new data (overfitting). We now evaluate both models on the **held-out test set**, which the model never saw during training.

This gives an honest estimate of how well the model generalizes.

In [ ]:
# Load and prepare the test data using the SAME preprocessing as training
df_test = pd.read_csv('insurance_test.csv')
df_test['smoker_enc'] = (df_test['smoker'] == 'yes').astype(int)

X_test_base = df_test[feature_cols]
y_test = df_test['charges']

# Build interaction features for test data (using test data values, not training)
X_test_interact = df_test[feature_cols].copy()
X_test_interact['age_x_bmi']         = df_test['age'] * df_test['bmi']
X_test_interact['age_x_smoker']      = df_test['age'] * df_test['smoker_enc']
X_test_interact['age_x_children']    = df_test['age'] * df_test['children']
X_test_interact['bmi_x_smoker']      = df_test['bmi'] * df_test['smoker_enc']
X_test_interact['bmi_x_children']    = df_test['bmi'] * df_test['children']
X_test_interact['smoker_x_children'] = df_test['smoker_enc'] * df_test['children']

# Evaluate both models on the test set
r2_base_test     = model_baseline.score(X_test_base, y_test)
mae_base_test    = mean_absolute_error(y_test, model_baseline.predict(X_test_base))
mse_base_test    = mean_squared_error(y_test, model_baseline.predict(X_test_base))

r2_inter_test    = model_interact.score(X_test_interact, y_test)
mae_inter_test   = mean_absolute_error(y_test, model_interact.predict(X_test_interact))
mse_inter_test   = mean_squared_error(y_test, model_interact.predict(X_test_interact))

print("Test Set Performance Comparison")
print(f"{'Model':<30} {'R²':>8} {'MAE':>12} {'MSE':>15}")
print("-" * 68)
print(f"{'Baseline MLR':<30} {r2_base_test:>8.4f} {mae_base_test:>12,.0f} {mse_base_test:>15,.0f}")
print(f"{'MLR with Interactions':<30} {r2_inter_test:>8.4f} {mae_inter_test:>12,.0f} {mse_inter_test:>15,.0f}")

### 3.5 Understanding the Evaluation Metrics

Three complementary metrics are used to assess regression model quality:

| Metric | Formula | Interpretation |
|---|---|---|
| **R²** (coefficient of determination) | 1 − (SS_res / SS_tot) | Proportion of variance explained; 1.0 = perfect, 0.0 = no better than mean |
| **MAE** (Mean Absolute Error) | mean(|y − ŷ|) | Average error in the original units (dollars here); easy to interpret |
| **MSE** (Mean Squared Error) | mean((y − ŷ)²) | Penalizes large errors more heavily; in squared units |

**RMSE** (Root Mean Squared Error) = √MSE brings MSE back to the original units and is more comparable to MAE.

In [ ]:
# Compute RMSE and display a final summary
rmse_base_test  = np.sqrt(mse_base_test)
rmse_inter_test = np.sqrt(mse_inter_test)

print("Complete Test Set Metrics")
print()
print(f"{'Metric':<10} {'Baseline MLR':>18} {'With Interactions':>20}")
print("-" * 52)
print(f"{'R²':<10} {r2_base_test:>18.4f} {r2_inter_test:>20.4f}")
print(f"{'MAE ($)':<10} {mae_base_test:>18,.2f} {mae_inter_test:>20,.2f}")
print(f"{'RMSE ($)':<10} {rmse_base_test:>18,.2f} {rmse_inter_test:>20,.2f}")

## 4. Interpreting the Coefficients

Each coefficient in a linear regression model represents how much the predicted outcome changes for a **one-unit increase** in that feature, holding all other features constant.

In [ ]:
# Coefficient summary for the interaction model
coef_df = pd.DataFrame({
    'Feature': list(X_interact.columns),
    'Coefficient': model_interact.coef_
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['crimson' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black', alpha=0.8)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Coefficient Value')
plt.title('Feature Coefficients — MLR with Interactions\n(Red = increases charges, Blue = decreases charges)')
plt.tight_layout()
plt.show()

## 5. Summary

This notebook demonstrated two core regression workflows:

**Simple Linear Regression (wheat yield)**
- Fit a line through one predictor variable and one outcome
- Used `coef_` and `intercept_` to interpret the fitted relationship
- Generated predictions for future years

**Multiple Linear Regression (insurance charges)**
- Encoded categorical variables (smoker → 0/1) for use in sklearn
- Started with a baseline 4-feature model
- Added all pairwise interaction terms to capture combined feature effects
- Evaluated models on both training and test data using R², MAE, and RMSE

**Key takeaways:**
- Always encode categorical variables before passing data to sklearn models
- Interaction terms can meaningfully improve R² when variables have combined effects
- Evaluate on a held-out test set — training performance is not a reliable estimate of generalization
- MAE and RMSE are more interpretable than MSE because they stay in the original units
- Smoker status was by far the strongest predictor of insurance charges in this dataset